# Regular Expressions

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
for parent in [current, *current.parents]:
    if (parent / '_config.yml').exists():
        project_root = parent  # ← Add project root, not chapters
        break
else:
    project_root = Path.cwd().parent.parent

sys.path.insert(0, str(project_root))

from shared import thinkpython, diagram, jupyturtle
from shared.download import download

# Register as top-level modules so direct imports work in subsequent cells
sys.modules['thinkpython'] = thinkpython
sys.modules['diagram'] = diagram
sys.modules['jupyturtle'] = jupyturtle

String methods allow you to search and manipulate text to a certain extent. A **regular expression** (regex) is a sequence of characters that defines a **search pattern** to search, match, and manipulate text in more powerful ways that go beyond simple string methods. For example:

| Task | Use |
|---|---|
| Check if string starts with 'http' | `str.startswith()` |
| Replace all spaces with underscores | `str.replace()` |
| Extract all email addresses from text | regex |
| Validate a phone number format | regex |
| Find words matching a complex pattern | regex |

The rule of thumb: if the pattern is **fixed and simple**, use string methods. If the pattern is **variable or complex**, use regex.

For example, to search a pattern in a text, we may use the `find()` string method and an index is returned if the pattern is found.

In [9]:
text = "I am Dracula; and I bid you welcome, Mr. Harker,\
    to my house."
pattern = 'Dracula'
text.find(pattern)

5

## Regex Escape Sequences

Before using `re` functions, we need to understand regex escapes.
In regex, the backslash `\` introduces special patterns.

Common escape sequences:

| Pattern | Meaning | Example match |
|---|---|---|
| `\d` | digit | `7` |
| `\w` | word character | `A`, `x`, `9`, `_` |
| `\s` | whitespace | space, tab |
| `\.` | literal dot | `.` |
| `\$` | literal dollar sign | `$` |

Why raw strings matter:
- Prefer regex patterns like `r'\d+'`.
- Raw strings avoid double escaping and make patterns easier to read.
- Without raw strings, you often need extra backslashes, like `'\\d+'`.

Use escapes for literal special regex characters too (for example `\.`, `\$`, `\?`).

In [ ]:
import re

demo = "Cost: $42.99 | User: alice_01 | gap	value"

print(re.findall(r'\d+', demo))      # digits
print(re.findall(r'\w+', demo))      # word chunks
print(re.findall(r'\$', demo))       # literal dollar sign
print(re.findall(r'\.', demo))       # literal dot

In [ ]:
### EXERCISE: Regex Escape Sequences
# Difficulty: Basic
s = "Price: $19.95, code=A_7, spaces here"
# 1. Extract all digit sequences
# 2. Extract all word tokens
# 3. Extract literal '$' and literal '.' matches
### Your code starts here:



### Your code ends here.

In [ ]:
import re

# Solution
s = "Price: $19.95, code=A_7, spaces here"
print(re.findall(r'\d+', s))
print(re.findall(r'\w+', s))
print(re.findall(r'\$', s))
print(re.findall(r'\.', s))

## The `re` Module

Python's built-in `re` module provides regex support. The five most commonly used regex functions are:

| Function | Description | Sample Syntax | Return |
|---|---|---|---|
| `re.search()` | Find first match anywhere in the string | `re.search(pattern, text)` | `Match` object or `None` |
| `re.match()` | Match only at the start of the string | `re.match(pattern, text)` | `Match` object or `None` |
| `re.findall()` | Find all matches; return as a list | `re.findall(pattern, text)` | `list` of strings |
| `re.sub()` | Find and replace matches | `re.sub(pattern, repl, text)` | `str` |
| `re.split()` | Split string on a pattern | `re.split(pattern, text)` | `list` of strings |

In [16]:
import re

text = "The price is $42.99 and the sale price is $29.99"
money_pattern = r'\$\d+(?:\.\d{2})?'

match = re.search(money_pattern, text)    # re.search() — find first match anywhere in the string
print("search:", match.group())           # $42.99

match = re.match(r'The', text)            # re.match() — match only at the START of the string
print("match:", match.group())            # The

prices = re.findall(money_pattern, text)  # re.findall() — find ALL matches, return as list
print("findall:", prices)                 # ['$42.99', '$29.99']

result = re.sub(money_pattern, 'PRICE', text)  # re.sub() — find and replace
print("sub:", result)                     # The price is PRICE and the sale price is PRICE

parts = re.split(r'\s+', "hello   world\tfoo")  # re.split() — split on a pattern
print("split:", parts)                    # ['hello', 'world', 'foo']

search: $42.99
match: The
findall: ['$42.99', '$29.99']
sub: The price is PRICE and the sale price is PRICE
split: ['hello', 'world', 'foo']


### `re.search()`

- returns: `re.search(pattern, text)` scans through `text` and returns 
  - a `Match` object for the **first** location where `pattern` is found. 
  - If the pattern is not found anywhere in the string, it returns `None`. 

- `Match`: A `Match` object has the following commonly used attributes and methods: 

| Attribute / Method | Description | Example |
|---|---|---|
| `.group()` | Returns the matched substring | `m.group()` → `'Dracula'` |
| `.start()` | Index where the match begins | `m.start()` → `5` |
| `.end()` | Index where the match ends | `m.end()` → `12` |
| `.span()` | Tuple of `(start, end)` | `m.span()` → `(5, 12)` |
| `.string` | The original string that was searched | `m.string` → `'I am Dracula...'` |

In [23]:
import re

text = "I am Dracula; and I bid you welcome, Mr. Harker, to my house."
pattern = 'Dracula'

result = re.search(pattern, text)     ### pattern: Dracula; text: the line
result

<re.Match object; span=(5, 12), match='Dracula'>

If the pattern appears in the text, `search` returns a `Match` object that contains the results of the search.
Among other information, it has a variable named `string` that contains the text that was searched.

In [24]:
result.string

'I am Dracula; and I bid you welcome, Mr. Harker, to my house.'

It also provides a method called `group` that returns the part of the text that **matched** the pattern.

In [29]:
result.group()

'Dracula'

:::{note}
`.group()` returns the **matched substring from the text** — the portion of the text that the pattern matched against. In simple cases like `re.search('Dracula', text)`, the match equals the pattern string. But with a regex like `r'\$[\d.]+'`, `.group()` would return something like `'$42.99'` — the actual text that matched, not the pattern expression itself.
:::

And it provides a method called `span` that returns the index in the text where the pattern starts and ends.

In [30]:
result.span()

(5, 12)

If the pattern doesn't appear in the text, the return value from `search` is `None`.

In [32]:
result = re.search('Count', text)
print(result)

None


So we can check whether the search was successful by checking whether the result is `None`.

In [9]:
result is None

True

In [ ]:
### EXERCISE: The `re` Module
# Difficulty: Basic
text = "Invoice totals: $15.00, $8.50, and $120"
# 1. Use re.findall() with a money regex to extract all amounts
# 2. Use re.sub() to replace each amount with the word PRICE
# 3. Print both results
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
import re

text = "Invoice totals: $15.00, $8.50, and $120"
money_pattern = r'\$\d+(?:\.\d{2})?'
print(re.findall(money_pattern, text))
print(re.sub(money_pattern, 'PRICE', text))

## Regex Syntax Essentials

Before writing larger patterns, it helps to know the core building blocks.

| Pattern | Meaning | Example Match |
|---|---|---|
| `\d` | digit | `7` |
| `\w` | word char (letter/digit/underscore) | `A`, `x`, `9`, `_` |
| `\s` | whitespace | space, tab |
| `[abc]` | one char from set | `a` |
| `[^abc]` | one char not in set | `z` |
| `+` | one or more | `aaa` |
| `*` | zero or more | `` (empty), `aaa` |
| `?` | optional (zero or one) | `u` in `colour` |
| `{m,n}` | repetition range | 2 to 4 times |

Use raw strings like `r'\d+'` for regex patterns so backslashes are interpreted correctly.

In [ ]:
sample = "User_42 logged in at 09:30 on 2026-03-11"

print(re.findall(r'\d+', sample))                  # all digit runs
print(re.findall(r'[A-Za-z_]+', sample))            # word-like alphabetic tokens
print(re.findall(r'\d{2}:\d{2}', sample))        # HH:MM time
print(re.findall(r'\d{4}-\d{2}-\d{2}', sample))  # YYYY-MM-DD date

In [ ]:
### EXERCISE: Regex Syntax Essentials
# Difficulty: Basic
s = "IDs: A12, B7, C999"
# 1. Extract all uppercase letters
# 2. Extract all digit sequences
# 3. Extract letter+digit tokens like A12, B7, C999
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
s = "IDs: A12, B7, C999"
print(re.findall(r'[A-Z]', s))
print(re.findall(r'\d+', s))
print(re.findall(r'[A-Z]\d+', s))

### `re.fullmatch()` for Validation

`re.fullmatch(pattern, text)` succeeds only if the **entire** string matches the pattern.
This is the right tool for validation tasks (IDs, simple emails, phone formats, etc.).

In [ ]:
employee_id_pattern = r'EMP-\d{4}'
ids = ['EMP-0001', 'EMP-12', 'AEMP-0001', 'EMP-12345']

for emp_id in ids:
    print(emp_id, bool(re.fullmatch(employee_id_pattern, emp_id)))

In [ ]:
### EXERCISE: Full String Validation
# Difficulty: Intermediate
codes = ['CS-101', 'MATH-240', 'CS101', 'EE-7']
# A valid course code must be: 2-4 uppercase letters, a dash, then 3 digits.
# 1. Write the regex pattern
# 2. Print each code with True/False using re.fullmatch
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
codes = ['CS-101', 'MATH-240', 'CS101', 'EE-7']
pattern = r'[A-Z]{2,4}-\d{3}'
for code_str in codes:
    print(code_str, bool(re.fullmatch(pattern, code_str)))

### Groups and Extraction

Parentheses create capture groups. You can extract parts of a match with `.group(1)`, `.group(2)`, etc.
Named groups can make patterns more readable.

In [ ]:
record = "OrderID=4821; Customer=Alice; Total=$39.50"
pattern = r'OrderID=(\d+); Customer=([A-Za-z]+); Total=\$(\d+(?:\.\d{2})?)'
m = re.search(pattern, record)

print(m.group(0))  # full match
print(m.group(1))  # order id
print(m.group(2))  # customer
print(m.group(3))  # total amount

In [ ]:
### EXERCISE: Capture Groups
# Difficulty: Intermediate
line = "name=Bob,age=27,dept=Sales"
# 1. Use one regex with 3 capture groups to extract name, age, dept
# 2. Print each extracted value on its own line
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
line = "name=Bob,age=27,dept=Sales"
m = re.search(r'name=([A-Za-z]+),age=(\d+),dept=([A-Za-z]+)', line)
print(m.group(1))
print(m.group(2))
print(m.group(3))

### Flags and Greedy vs Non-greedy

Flags change matching behavior:
- `re.IGNORECASE`: case-insensitive matching
- `re.MULTILINE`: `^` and `$` apply per line
- `re.DOTALL`: `.` also matches newline

Quantifiers like `*` and `+` are greedy by default. Add `?` to make them non-greedy.

In [ ]:
text_block = """Title: Notes
Email: ALICE@example.com
Email: bob@Example.org"""

# IGNORECASE
emails = re.findall(r'[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}', text_block, flags=re.IGNORECASE)
print(emails)

# Greedy vs non-greedy on tags
html = "<b>bold</b><i>italic</i>"
print(re.findall(r'<.*>', html))     # greedy
print(re.findall(r'<.*?>', html))    # non-greedy

In [ ]:
### EXERCISE: Flags and Quantifiers
# Difficulty: Challenge
text_block = """Task: clean logs
ERROR: Disk full
error: retry failed
INFO: done"""
# 1. Extract all lines that start with 'error' (case-insensitive) using MULTILINE
# 2. From '<x>1</x><x>2</x>', extract tags non-greedily
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
text_block = """Task: clean logs
ERROR: Disk full
error: retry failed
INFO: done"""

errs = re.findall(r'^error:.*$', text_block, flags=re.IGNORECASE | re.MULTILINE)
print(errs)

print(re.findall(r'<.*?>', '<x>1</x><x>2</x>'))

## Download and clean the text

Before we can search the text of *Dracula*, we need to download it from Project Gutenberg and remove the header and footer information.

We'll download the Dracula text from Project Gutenberg and save it to the `data` folder. Then we'll clean the file and save the cleaned version in the same folder. All subsequent analysis will use these files.

In [10]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = project_root / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

# Download Dracula text to the project data folder
url = 'https://www.gutenberg.org/files/345/345-0.txt'
raw_path = data_dir / 'pg345.txt'
clean_path = data_dir / 'pg345_cleaned.txt'
if not raw_path.exists():
    urlretrieve(url, raw_path)
    print('Downloaded Dracula to', raw_path)
else:
    print('Dracula already downloaded:', raw_path)

Dracula already downloaded: ../../data/pg345.txt


In [11]:
# download('https://www.gutenberg.org/cache/epub/345/pg345.txt');

In [12]:
def clean_file(infile, outfile):
    """Read infile, write to outfile skipping special lines."""
    with open(infile, encoding='utf8') as fin, open(outfile, 'w', encoding='utf8') as fout:
        for line in fin:
            if not is_special_line(line):
                fout.write(line)

In [13]:
# def clean_file(input_file, output_file):
#     reader = open(input_file, encoding='utf-8')
#     writer = open(output_file, 'w')

#     for line in reader:
#         if is_special_line(line):
#             break

#     for line in reader:
#         if is_special_line(line):
#             break
#         writer.write(line)
        
#     reader.close()
#     writer.close()

In [14]:
def is_special_line(line):
    """Return True if the line is a header/footer or empty."""
    return (line.startswith('***') or line.strip() == '' or
            line.startswith('End of the Project Gutenberg'))

In [15]:
# def is_special_line(line):
#     return line.strip().startswith('*** ')

In [16]:
# Clean the Dracula text and save to data/pg345_cleaned.txt
clean_file(raw_path, clean_path)
print('Cleaned file saved to', clean_path)

Cleaned file saved to data/pg345_cleaned.txt


Putting all that together, here's a function that loops through the lines in the book until it finds one that matches the given pattern, and returns the `Match` object.

In [17]:
def find_first(pattern, path=clean_path):
    with open(path, encoding='utf8') as f:
        for line in f:
            result = re.search(pattern, line)
            if result is not None:
                return result

We can use it to find the first mention of a character.

In [18]:
result = find_first('Harker')
result.string

'CHAPTER I. Jonathan Harker’s Journal\n'

For this example, we didn't have to use regular expressions -- we could have done the same thing more easily with the `in` operator.
But regular expressions can do things the `in` operator cannot.

For example, if the pattern includes the vertical bar character, `'|'`, it can match either the sequence on the left or the sequence on the right.
Suppose we want to find the first mention of Mina Murray in the book, but we are not sure whether she is referred to by first name or last.
We can use the following pattern, which matches either name.

In [19]:
pattern = 'Mina|Murray'
result = find_first(pattern)
result.string

'CHAPTER V. Letters—Lucy and Mina\n'

We can use a pattern like this to see how many times a character is mentioned by either name.
Here's a function that loops through the book and counts the number of lines that match the given pattern.

In [20]:
def count_matches(pattern, path=clean_path):
    count = 0
    with open(path, encoding='utf8') as f:
        for line in f:
            result = re.search(pattern, line)
            if result is not None:
                count += 1
    return count

Now let's see how many times Mina is mentioned.

In [21]:
count_matches('Mina|Murray')

229

The special character `'^'` matches the beginning of a string, so we can find a line that starts with a given pattern.

In [22]:
result = find_first('^Dracula')
result.string

'Dracula, jumping to his feet, said:--\n'

And the special character `'$'` matches the end of a string, so we can find a line that ends with a given pattern (ignoring the newline at the end).

In [23]:
result = find_first('Harker$')
result.string

'by five o’clock, we must start off; for it won’t do to leave Mrs. Harker\n'

In [ ]:
### EXERCISE: Download and Clean Text
# Difficulty: Intermediate
# 1. Use raw_path and clean_path to print whether each file exists
# 2. If clean_path does not exist, run clean_file(raw_path, clean_path)
# 3. Print the size (in bytes) of clean_path
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
print(raw_path.exists(), clean_path.exists())
if not clean_path.exists():
    clean_file(raw_path, clean_path)
print(clean_path.stat().st_size)

### String substitution

Bram Stoker was born in Ireland, and when *Dracula* was published in 1897, he was living in England.
So we would expect him to use the British spelling of words like "centre" and "colour".
To check, we can use the following pattern, which matches either "centre" or the American spelling "center".

In [24]:
pattern = 'cent(er|re)'

In this pattern, the parentheses enclose the part of the pattern the vertical bar applies to.
So this pattern matches a sequence that starts with `'cent'` and ends with either `'er'` or `'re'`.

In [25]:
result = find_first(pattern)
result.string

'horseshoe of the Carpathians, as if it were the centre of some sort of\n'

As expected, he used the British spelling.

We can also check whether he used the British spelling of "colour".
The following pattern uses the special character `'?'`, which means that the previous character is optional.

In [26]:
pattern = 'colou?r'

This pattern matches either "colour" with the `'u'` or "color" without it.

In [27]:
result = find_first(pattern)
line = result.string
line

'undergarment with long double apron, front, and back, of coloured stuff\n'

Again, as expected, he used the British spelling.

Now suppose we want to produce an edition of the book with American spellings.
We can use the `sub` function in the `re` module, which does **string substitution**.

In [28]:
re.sub(pattern, 'color', line)

'undergarment with long double apron, front, and back, of colored stuff\n'

In [ ]:
### EXERCISE: String Substitution
# Difficulty: Intermediate
sample = "The colour of the city centre changed overnight."
# 1. Replace British spellings with American spellings using regex:
#    colour -> color, centre -> center
# 2. Print the transformed sentence
### Your code starts here:



### Your code ends here.

In [ ]:
# Solution
import re

sample = "The colour of the city centre changed overnight."
sample = re.sub(r'colou?r', 'color', sample)
sample = re.sub(r'cent(er|re)', 'center', sample)
print(sample)

The first argument is the pattern we want to find and replace, the second is what we want to replace it with, and the third is the string we want to search.
In the result, you can see that "colour" has been replaced with "color".

In [29]:
# I used this function to search for lines to use as examples

def all_matches(pattern, path=clean_path):
    with open(path, encoding='utf8') as f:
        for line in f:
            result = re.search(pattern, line)
            if result:
                print(line.strip())

In [30]:
### e.g., 

all_matches('weather')

weather. As I stood, the driver jumped again into his seat and shook the
weatherworn, was still complete; but it was evidently many a day since
it is a buoy with a bell, which swings in bad weather, and sends in a
am awakened by her moving about the room. Fortunately, the weather is so
learn the weather signs. To-day is a grey day, and the sun as I write is
experienced here, with results both strange and unique. The weather had
kept watch on weather signs from the East Cliff, foretold in an emphatic
_22 July_.--Rough weather last three days, and all hands busy with
weather. Passed Gibralter and out through Straits. All well.
and entering on the Bay of Biscay with wild weather ahead, and yet last
weather influences as we know that the Count can bring to bear; and if
that I am fully armed as there may be wolves; the weather is getting


In [31]:
# Here's the pattern I used (which uses some features we haven't seen)

# names = r'(?<!\.\s)[A-Z][a-zA-Z]+'

# all_matches(names)